### **1. Import & Load Dat**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import numpy as np
from sklearn.model_selection import train_test_split
import ast
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "listings.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PATH = PROJECT_ROOT / "data" / "listings.csv"
df = pd.read_csv(PATH)

# Display all the columns
pd.set_option('display.max_columns', None)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
exact_dup_count = df.duplicated().sum()
print(exact_dup_count)

In [ ]:
df["id"].duplicated().sum()

### **2. Initial Data Cleaning & Column Dropping**

In [ ]:
# Turn the string "price" data into numeric
df["price"] = (
    df["price"]
    .str.replace(r"[^0-9.]", "", regex=True)
    .astype(float)
)

In [ ]:
# Convert percentage strings to floats (fake strings)
rates = ["host_response_rate", "host_acceptance_rate"]
for col in rates:
    df[col] = df[col].astype(str).str.replace("%", "").astype(float) / 100

In [ ]:
# Check for missing values
df.isnull().mean().mul(100).sort_values(ascending=False)

In [ ]:
# Drop all the rows whose price value is NaN, imputation is not an option since they're the target values
df = df.dropna(subset=["price"]).copy()

In [ ]:
# Drop useless and leakage columns

cols_to_drop = [
    "id",
    "listing_url",
    "scrape_id",
    "source",
    "picture_url",
    "host_id",
    "host_url",
    "host_thumbnail_url",
    "host_picture_url",
    "calendar_updated",          # all null
    "calendar_last_scraped",
    "estimated_revenue_l365d",   # leakage
    "estimated_occupancy_l365d", # leakage

    # text fields
    "name",
    "description",
    "neighborhood_overview",
    "host_about",
    "bathrooms_text",

    # redundant/free-text versions
    "host_name",          
    "host_verifications", 
]

df = df.drop(columns=cols_to_drop)

### **3. Train/Test Split**

The dataset was split before EDA to keep the test set untouched until final evaluation.

A temporary `price_bin` column was created with `pd.qcut` to group prices into quantile-based bins. These bins were used for stratification so that train and test sets keep a similar price distribution.

The split was performed as:

- 80% training
- 20% test
- `random_state=42`
- stratified on `price_bin`

After splitting, `price_bin` was removed from both sets because it was only a helper variable for the split.


In [ ]:
# Create helper bins (quantiles) for stratified train/test split on price
df["price_bin"] = pd.qcut(df["price"], q=5, duplicates="drop")

In [ ]:
# Split data into 80% trainig and 20% test sets using stratification on price bins.
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["price_bin"]
)

In [ ]:
# Remove the price bins from each set. 
df_train = df_train.drop(columns=["price_bin"])
df_test = df_test.drop(columns=["price_bin"])

### **4. Exploratory Data Analysis**

This section explores the **training set** after the initial cleaning and split, with the goal of understanding the variables that may explain Airbnb listing prices in the **Basque Country**. Since `price` is strongly right-skewed, the analysis focuses mainly on **`log_price`** as a more stable target for interpretation and later modeling.

The EDA is organized around the main groups of features in the dataset. First, I examine **numerical variables** through descriptive statistics, distributions, correlations, and transformations such as clipping and logarithms. Then, I explore **geographical patterns**, **categorical variables**, **host tenure**, and **amenities** to identify which characteristics appear most related to price differences across listings. This section also helps define a practical **missing value strategy** and supports the final feature selection used in the modeling stage.


In [ ]:
df_eda = df_train.copy()

In [ ]:
df_eda.shape

In [ ]:
df_eda.info()

In [ ]:
df_eda.head()

In [ ]:
df_eda.describe()

The price data is right-skewed, there are outliers which can distort the model, so we may try to apply log transformation on prices to make the distribution more balanced. 

In [ ]:
# Check the distribution of the target variable with a histogram. 
df_eda["price"].hist(bins=50, figsize=(10, 6))

In [ ]:
# Check the target variable with a histogram after log transformation. It's more normal now.
np.log1p(df_eda["price"]).hist(bins=50, figsize=(10, 6))

As we can see, the large values are compressed, skewness reduced and the price data is more normal now. 

In [ ]:
lower = 0.01
upper = 0.99

# Visualize clipped + logged price distribution
price_lower_bound = np.log1p(df_eda["price"]).quantile(lower)
price_upper_bound = np.log1p(df_eda["price"]).quantile(upper)
clipped_logged_price = np.clip(np.log1p(df_eda["price"]), price_lower_bound, price_upper_bound)
clipped_logged_price.hist(bins=50, figsize=(10, 6))

The target variable `price` was highly right-skewed, so its distribution was inspected on the log scale.

To reduce the effect of extreme values during analysis, `log1p(price)` was clipped between the 1st and 99th percentiles and then plotted as a histogram.

##### **4.a. Analysis of Numerical Features**

* **Histograms**

In [ ]:
cols_host = ["host_response_rate", "host_acceptance_rate"]

# Convert to numeric percentages
for col in cols_host:
    df[f"{col}_num"] = pd.to_numeric(
        df[col].astype(str).str.replace("%", "", regex=False),
        errors="coerce"
    )

df[[f"{col}_num" for col in cols_host]].describe()

In [ ]:
# Clip + log transform the price column and add it to df_eda for further analysis.
df_eda["log_price"] = np.clip(np.log1p(df_eda["price"]), price_lower_bound, price_upper_bound)

# Plot the histograms for all numeric columns in df_eda to 
# check their distributions. We can apply transformations later if needed.
df_eda.select_dtypes(include="number").hist(
    bins=50, 
    figsize=(35, 20), 
    grid=True
)
plt.tight_layout()
plt.show()

df_eda = df_eda.drop(columns=["price"])

* **Correlation Check**

In [ ]:
numeric_cols = df_eda.select_dtypes(include="number").columns

In [ ]:
corr_matrix = df_eda[numeric_cols].corr()

In [ ]:
plt.figure(figsize=(12,10))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    annot=False
)

plt.title("Correlation Heatmap of Numerical Features")
plt.show()

In [ ]:
df_eda[numeric_cols].corrwith(df_eda["log_price"]).sort_values(ascending=False)

* Scatterplot

In [ ]:
attributes = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "review_scores_location",
    "log_price" 
]

scatter_matrix(
    df_eda[attributes],
    figsize=(15, 11),
    alpha=0.4
)

* **Checking the log transformation candidates**

In [ ]:
log_candidates = pd.DataFrame({
    "log_min_nights": np.log1p(df_eda["minimum_nights"]),
    "log_max_nights": np.log1p(df_eda["maximum_nights"]),
    "log_calculated_host_listings_count": np.log1p(df_eda["calculated_host_listings_count"]),
    "log_reviews_per_month": np.log1p(df_eda["reviews_per_month"]),
    "log_number_of_reviews": np.log1p(df_eda["number_of_reviews"]),
    "log_host_response_rate": np.log1p(df_eda["host_response_rate"]),
    "log_host_acceptance_rate": np.log1p(df_eda["host_acceptance_rate"]),
    "log_accommodates": np.log1p(df_eda["accommodates"]),
    "log_bedrooms": np.log1p(df_eda["bedrooms"]),
    "log_beds": np.log1p(df_eda["beds"]),
    "log_bathrooms": np.log1p(df_eda["bathrooms"]),
})

log_candidates.hist(bins=50, figsize=(10, 6))
plt.tight_layout()
plt.show()

In [ ]:
log_candidates.corrwith(df_eda["log_price"]).sort_values(ascending=False)

In [ ]:
log_candidates.describe()

* **Checking the "clip" (or log+clip) candidates**

In [ ]:
clip_cols = [
    "beds",
    "minimum_nights",
    "maximum_nights",
    "host_acceptance_rate",
    "bedrooms",
    "bathrooms",
    "beds", "accommodates"
]

clip_candidates = pd.DataFrame()

for col in clip_cols:
    lower = df_eda[col].quantile(0.01)
    upper = df_eda[col].quantile(0.99)
    clipped = df_eda[col].clip(lower=lower, upper=upper)

    if col in ["minimum_nights", "maximum_nights"]:
        clip_candidates[f"log_{col}_clipped"] = np.log1p(clipped)
    else:
        clip_candidates[f"{col}_clipped"] = clipped


In [ ]:
clip_candidates.corrwith(df_eda["log_price"]).sort_values(ascending=False)

In [ ]:
clip_candidates.hist(bins=50, figsize=(10, 6))
plt.tight_layout()
plt.show()


* **Choice of Log and Clipped Features**

    Skewed numeric variables were tested in transformed versions and compared against `log_price` using simple correlation checks. The goal was to keep only the transformations that looked more useful than the raw form.
    
    **Clipped features**
    The following variables were selected for clipping because their distributions contained extreme values and clipping produced a more    reasonable version for modeling:
    
    - `beds`
    - `bathrooms`
    - `bedrooms`
    - `minimum_nights`
    - `maximum_nights`
    - `host_acceptance_rate_num`
    
    **Log-transformed features**
    The following variables were selected for log transformation because their logged versions showed a better or more usable relationship  with `log_price` in the correlation checks:
    
    - `minimum_nights`
    - `accommodates`
    - `maximum_nights`
    - `number_of_reviews`
    - `reviews_per_month`
    
    **Note**
    Some variables, such as `minimum_nights` and `maximum_nights`, were included in both groups. This reflects that both clipping and log transformation were considered useful for handling skewness and extreme values in those features. 
    Although some of these variables remain correlated (for example capacity-related variables and availability metrics), this doesnot pose a significant issue because the final model will use **Ridge Regression**. Ridge regression applies L2regularization, which shrinks coefficients of correlated predictors and mitigates multicollinearity without requiring strictfeature independence.


* **Analysis of Geographical Features**

In [ ]:
df_eda.plot(kind="scatter", x="longitude", y="latitude")

In [ ]:
df_eda.plot(kind="scatter", x="longitude", y="latitude", alpha=0.1)

In [ ]:
df_eda.plot(
    kind="scatter",
    x="longitude",
    y="latitude",
    alpha=0.3,
    c="log_price", 
    cmap="viridis",
    colorbar=True,
    figsize=(10,7)
)

plt.title("Airbnb Prices Across the Basque Country")
plt.show()

In [ ]:
# 1. Define a Basque Country reference point for one distance feature
BASQUE_COUNTRY_REFERENCE_LAT = 43.2630
BASQUE_COUNTRY_REFERENCE_LON = -2.9350

# 2. Calculate the basic Euclidean distance
df_eda["distance_to_basque_country_reference"] = np.sqrt(
    (df_eda["latitude"] - BASQUE_COUNTRY_REFERENCE_LAT)**2 + 
    (df_eda["longitude"] - BASQUE_COUNTRY_REFERENCE_LON)**2
)

df_eda[["distance_to_basque_country_reference", "longitude", "latitude"]].corrwith(df_eda["log_price"])

In [ ]:
DONOSTIA_LAT = 43.3183
DONOSTIA_LON = -1.9812

df_eda["distance_to_donostia"] = np.sqrt(
    (df_eda["latitude"] - DONOSTIA_LAT)**2 + 
    (df_eda["longitude"] - DONOSTIA_LON)**2
)

df_eda[["distance_to_donostia", "longitude", "latitude"]].corrwith(df_eda["log_price"])

In [ ]:
VITORIA_LAT = 42.8467
VITORIA_LON = -2.6726

df_eda["distance_to_vitoria"] = np.sqrt(
    (df_eda["latitude"] - VITORIA_LAT)**2 + 
    (df_eda["longitude"] - VITORIA_LON)**2
)

df_eda[["distance_to_vitoria", "longitude", "latitude"]].corrwith(df_eda["log_price"])

In [ ]:
COAST_LAT = 43.3623
COAST_LON = -3.0136

df_eda["distance_to_coast"] = np.sqrt(
    (df_eda["latitude"] - COAST_LAT)**2 + 
    (df_eda["longitude"] - COAST_LON)**2
)

df_eda[["distance_to_coast", "longitude", "latitude"]].corrwith(df_eda["log_price"])

* Geographic Feature Engineering

    Since the dataset contains listings across the entire **Basque Country**, using raw geographic coordinates  (`latitude`, `longitude`) in a **linear model such as Ridge Regression** is not ideal. Linear models cannot easily   capture spatial relationships directly from coordinates.

    Instead, I engineered **distance-based features** that better represent location effects on price:

    - `distance_to_basque_country_reference`
    - `distance_to_donostia`
    - `distance_to_vitoria`
    - `distance_to_coast`

    These were computed using the Euclidean distance between each listing's coordinates and the reference location.

    Correlation checks show that these distance features have **stronger and more interpretable relationships with  `log_price`** than raw coordinates, making them more suitable inputs for the model.


* List of numerical columns to keep:
    - log_price
    - host_listings_count
    - host_total_listings_count
    - log_accommodates
    - bathrooms_clipped
    - bedrooms_clipped
    - beds_clipped
    - log_minimum_nights_clipped
    - log_maximum_nights_clipped
    - availability_30
    - availability_90
    - availability_365
    - log_number_of_reviews
    - log_reviews_per_month
    - review_scores_rating
    - review_scores_cleanliness
    - review_scores_location
    - calculated_host_listings_count_entire_homes
    - calculated_host_listings_count_private_rooms
    - calculated_host_listings_count_shared_rooms
    - host_acceptance_rate_clipped
    - distance_to_coast
    - distance_to_vitoria
    - distance_to_donostia
    - distance_to_center
    - longitude
    - latitude

##### **4.b. Analysis of Categorical Features**

In [ ]:
# Get a list of all categorical columns
categorical_cols = df_eda.select_dtypes(include=["object", "string"]).columns

print(len(categorical_cols))
categorical_cols

In [ ]:
df_eda[categorical_cols].head()

In [ ]:
cat_cols_to_drop = [
    "first_review", 
    "last_review", "license",
    "host_location", "host_neighbourhood"
]

df_eda = df_eda.drop(columns=cat_cols_to_drop, errors="ignore")

In [ ]:
print(f"Unique property types: {df_eda['property_type'].nunique()}")
print(f"Unique neighborhood groups: {df_eda['neighbourhood_group_cleansed'].nunique()}")
print(f"Unique neighborhoods: {df_eda['neighbourhood'].nunique()}")
print(f"Unique neighborhood cleansed: {df_eda['neighbourhood_cleansed'].nunique()}")
print(f"Unique host response time types: {df_eda['host_response_time'].nunique()}")

In [ ]:
# Calculate the percentage of total listings for each property type
prop_percentages = df_eda["property_type"].value_counts(normalize=True) * 100
print(prop_percentages.head(15).round(2).astype(str) + '%')

In [ ]:
# Define how many top categories you want to keep
TOP_K = 10

# PROPERTY TYPE
top_properties = df_eda["property_type"].value_counts().nlargest(TOP_K).index

df_eda["property_type_clean"] = df_eda["property_type"].where(
    df_eda["property_type"].isin(top_properties),
    "Other"
)

# NEIGHBOURHOOD CLEANSED
top_neighbourhoods_cleansed = (
    df_eda["neighbourhood_cleansed"]
    .value_counts()
    .nlargest(TOP_K)
    .index
)

df_eda["neighbourhood_cleansed_clean"] = df_eda["neighbourhood_cleansed"].where(
    df_eda["neighbourhood_cleansed"].isin(top_neighbourhoods_cleansed),
    "Other"
)

# NEIGHBOURHOOD
top_neighbourhoods = (
    df_eda["neighbourhood"]
    .value_counts()
    .nlargest(TOP_K)
    .index
)

df_eda["neighbourhood_clean"] = df_eda["neighbourhood"].where(
    df_eda["neighbourhood"].isin(top_neighbourhoods),
    "Other"
)

# Prove it worked
print("Property types:")
print(df_eda["property_type_clean"].value_counts())

print("\nNeighbourhood cleansed:")
print(df_eda["neighbourhood_cleansed_clean"].value_counts())

print("\nNeighbourhood:")
print(df_eda["neighbourhood_clean"].value_counts())

* Why `neighbourhood_cleansed` is used instead of other neighbourhood features

    The dataset contains three location-related variables: `neighbourhood`, `neighbourhood_group_cleansed`, and     `neighbourhood_cleansed`.
    
    - `neighbourhood` contains inconsistent text strings (e.g., different free-text location spellings),     which creates redundant categories and noise.
    - `neighbourhood_group_cleansed` is too coarse, containing only a few broad geographic groups.
    - `neighbourhood_cleansed` provides standardized municipality names (e.g., Basque Country municipalities such as Getxo and Donostia-San Sebastián),  giving a better balance between **data quality and geographic detail**.
    
    Therefore, `neighbourhood_cleansed` is retained for feature engineering while the other two variables are dropped.

In [ ]:
df_eda.groupby("neighbourhood_cleansed_clean")["log_price"].median().sort_values()

* What the `neighbourhood_cleansed` – `log_price` analysis shows

    The median `log_price` was computed for each `neighbourhood_cleansed`.
    
    The results show large variation in prices across locations, with median `log_price` values ranging roughly from **3.   3 to over 6**. After converting from the logarithmic scale, this corresponds to substantial differences in actual  listing prices.
    
    This indicates that **location is a strong driver of Airbnb prices**, confirming the importance of including a  reliable geographic feature in the model.

In [ ]:
loc_cols_to_drop = [
   "neighbourhood", "neighbourhood_group_cleansed"
]

# Drop them from your EDA dataframe
df_eda = df_eda.drop(columns=loc_cols_to_drop, errors="ignore")

In [ ]:
cat_cols = [
    "property_type_clean",
    "room_type",
    "instant_bookable",
    "neighbourhood_cleansed_clean",
    "host_is_superhost",
    "host_has_profile_pic",
    "host_response_time"
]

n_plot_cols = 2
n_plot_rows = (len(cat_cols) + n_plot_cols - 1) // n_plot_cols

fig, axes = plt.subplots(n_plot_rows, n_plot_cols, figsize=(15, 5 * n_plot_rows))
axes = np.atleast_1d(axes).flatten()

for i, col in enumerate(cat_cols):
    sns.boxplot(data=df_eda, x=col, y="log_price", ax=axes[i])
    axes[i].set_title(f"Price Distribution by {col}")
    axes[i].tick_params(axis="x", rotation=45)

for ax in axes[len(cat_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()


* Categorical Feature Exploration

    These boxplots show how **log-transformed price (`log_price`)** varies across key categorical variables.

    - **property_type_clean**: Different property types show noticeable differences in price distribution. Entire homes     and cottages tend to have higher median prices than private rooms or hostel-type listings.

    - **room_type**: This variable has a clear relationship with price. Entire homes/apartments have the highest median     prices, while shared rooms have the lowest.

    - **instant_bookable**: Listings that can be instantly booked show a slightly higher median price, though the   difference appears relatively small.

    - **neighbourhood_cleansed_clean**: Price distributions vary across locations, suggesting that neighbourhood plays an   important role in determining listing price.

    Overall, **room type and neighbourhood appear to be the strongest categorical signals for price**, while **instant  booking status shows a weaker effect**.

##### **4.c. Host Tenure Analysis**

In this subsection, I analyze **host tenure** by converting the date variables into a usable datetime format and calculating how long each host has been active on the platform. From this, I create features such as `host_experience_days` and `host_experience_years`, which capture the host’s level of experience at the time of data collection.

The aim is to examine whether more experienced hosts tend to charge different prices than newer hosts. I summarize the distribution of host experience and check its relationship with `log_price` to assess whether host tenure may be a useful predictor in the final model.


In [ ]:
df_eda["last_scraped"] = pd.to_datetime(df["last_scraped"], errors="coerce")
df_eda["last_scraped"].max()

In [ ]:
df_eda["host_since"] = pd.to_datetime(df_eda["host_since"], errors="coerce")
reference_date = pd.Timestamp("2025-09-30")
df_eda["host_experience_days"] = (reference_date - df_eda["host_since"]).dt.days
df_eda["host_experience_years"] = df_eda["host_experience_days"] / 365.25

In [ ]:
date_cols = ["host_experience_days", "host_experience_years"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, date_cols):
    sns.histplot(df_eda[col], bins=30, ax=ax)
    ax.set_title(col)

plt.tight_layout()

In [ ]:
df_eda["host_experience_days"].describe()

In [ ]:
df_eda[["host_experience_days", "log_price"]].corr()

##### **4.d. The Analysis of Amenities**

In this subsection, I explore the **amenities** offered by each listing to understand whether certain features are associated with higher prices. Since amenities are stored as text-based lists, they first need to be parsed and transformed into a structured format before they can be analyzed properly.

The goal is to identify which amenities are most common, which ones appear more often in higher-priced listings, and whether some features may act as indicators of more premium properties. This analysis helps uncover additional signals beyond location, capacity, and host characteristics that may improve the final model.


In [ ]:
# preview the raw amenities
print(df_eda["amenities"].head(5))

In [ ]:
# Parse amenities
def parse_amenities(value):
    if pd.isna(value) or value == "":
        return []
    if isinstance(value, list):
        parsed = value
    else:
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    return [str(amenity).strip().strip('"') for amenity in parsed if str(amenity).strip()]

amenities_lists = df_eda["amenities"].apply(parse_amenities)
amenities_sets = amenities_lists.apply(set)

# Count amenities
amenities_counts = pd.Series(
    [amenity for sublist in amenities_lists for amenity in sublist]
).value_counts()

print("Total unique amenities:", amenities_counts.shape[0])

In [ ]:
# Define a minimum share threshold (e.g., 5% of listings) to select amenities
min_share = 0.05
min_count = int(len(df_eda) * min_share)

selected_amenities = amenities_counts[amenities_counts >= min_count].index.tolist()
len(selected_amenities)

In [ ]:
# Analyze the price impact of each selected amenity by comparing the median 
# log price of listings with and without the amenity.

# Number of listings in the EDA dataset
n_listings = len(df_eda)

# Keep only amenities that appear often enough to be meaningful
MIN_SHARE = 0.05   # at least 5% of listings
MIN_COUNT = int(n_listings * MIN_SHARE)

candidate_amenities = amenities_counts[amenities_counts >= MIN_COUNT].index.tolist()

# Build relevance table
results = []

for amenity in candidate_amenities:
    has_amenity = amenities_sets.apply(lambda s: amenity in s)

    median_with = df_eda.loc[has_amenity, "log_price"].median()
    median_without = df_eda.loc[~has_amenity, "log_price"].median()

    diff_log = median_with - median_without
    approx_pct_diff = (np.exp(diff_log) - 1) * 100

    results.append({
        "amenity": amenity,
        "count": has_amenity.sum(),
        "share_listings_pct": has_amenity.mean() * 100,
        "median_log_price_with": median_with,
        "median_log_price_without": median_without,
        "diff_log_price": diff_log,
        "approx_price_diff_pct": approx_pct_diff
    })

amenity_impact = pd.DataFrame(results)

# Rank by strongest price effect
amenity_impact = amenity_impact.sort_values(
    by="diff_log_price",
    key=lambda s: s.abs(),
    ascending=False
).reset_index(drop=True)

# Show the most price-relevant amenities
amenity_impact.head(20)


In [ ]:
display(amenities_counts.head(100))

In [ ]:
# Analyze the amenities of the most expensive listings to see if there are any common features.

top_expensive_idx = df_eda.sort_values("log_price", ascending=False).head(50).index

top_expensive_amenities = amenities_lists.loc[top_expensive_idx]

top_expensive_counts = pd.Series(
    [amenity for sublist in top_expensive_amenities for amenity in sublist]
).value_counts()

display(top_expensive_counts.head(100))

In [ ]:
# Check which of the most common amenities among the expensive listings are not in 
# the selected amenities list.

top_expensive_rare_counts = pd.Series(
    [
        amenity
        for sublist in top_expensive_amenities
        for amenity in sublist
        if amenity not in selected_amenities
    ]
).value_counts()

display(top_expensive_rare_counts.head(50))

In [ ]:
# Define a list of luxury-related keywords to check for in the amenities.
luxury_keywords = [
    "view",
    "beachfront",
    "beach access",
    "pool",
    "hot tub",
    "fireplace",
    "gym",
    "ev charger"
]
# Create a binary feature indicating whether a listing has any luxury-related amenity.
df_eda["has_luxury_amenity"] = amenities_lists.apply(
    lambda row: int(any(
        keyword in amenity.lower()
        for amenity in row
        for keyword in luxury_keywords
    ))
)
# Create a count feature for the number of luxury-related amenities.
df_eda["luxury_amenity_count"] = amenities_lists.apply(
    lambda row: sum(
        any(keyword in amenity.lower() for keyword in luxury_keywords)
        for amenity in row
    )
)

In [ ]:
# Check the correlation of these new features with log_price.
display(
    df_eda[["log_price", "has_luxury_amenity", "luxury_amenity_count"]]
    .corr(numeric_only=True)
)

In [ ]:
# Bin log_price into quartiles and analyze the average presence of luxury amenities 
# in each price bin.
df_eda["price_bin"] = pd.qcut(
    df_eda["log_price"],
    q=4,
    labels=["low", "mid_low", "mid_high", "high"]
)

display(
    df_eda.groupby("price_bin")[["has_luxury_amenity", "luxury_amenity_count"]]
    .agg(["mean", "median"])
)

Overall, the amenities analysis shows that not all amenities contribute equally to explaining price differences. While many common amenities are widely available across listings and offer limited predictive value, a smaller group of more distinctive features appears to be associated with higher-priced properties and may serve as indicators of quality or premium positioning.

This step helps move the amenities variable from an unstructured text field to a more meaningful source of information for modeling. The insights gained here support the selection of the most relevant amenities-related features and help ensure that only useful signals are carried forward into the preprocessing and modeling stages.


##### **4.d. Missing Value Strategy**

In [ ]:
# Check the percentage of missing values in each column to decide on imputation or dropping.
missing_cols = df_eda.isnull().mean().mul(100).sort_values(ascending=False)
missing_cols[missing_cols > 0]

* Missing Value Strategy

    - Missing values were analyzed before preprocessing. Most missing values occur in review-related variables  (~9%), which correspond to listings with no reviews yet. These values will therefore be filled with **0**    to represent the absence of reviews.
    
    - Numeric listing characteristics such as `beds`, `bathrooms`, and `bedrooms` have extremely small missing  proportions (<1%) and will also be imputed with the **median**.